# f696 Stage1 因子到资金动作的区间对照

冻结的只读诊断；复用原生 Qlib Recorder、Qlib 日历和 SOXX 收盘来源。事后好坏分组仅用于理解失败链，不生成交易规则。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

root = Path.cwd()
card = root / "research/stage1_f696_factor_interval_delivery_20261024"
plan_path = card / "plan.json"
plan = json.loads(plan_path.read_text())
assert plan["frozen"] and all(v == 0 for v in plan["budget"].values())
src = plan["source"]
source_card = root / src["card"]
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(source_card / "summary.json") == src["summary_sha256"]
summary = json.loads((source_card / "summary.json").read_text())
assert summary["nas_readback"]["verified"] and summary["nas_readback"]["sha256"] == src["nas_sha256"]
assert summary["nas_evidence"]["sha256"] == src["nas_sha256"] and summary["nas_evidence"]["size_bytes"] == src["nas_size_bytes"]
artifact_root = source_card / "qlib_recorder" / src["experiment_id"] / src["recorder_id"] / "artifacts"
assert sha(artifact_root / "analysis/daily") == src["daily_sha256"]
assert sha(root / src["provider"] / "qlib_store_manifest.json") == src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root / src["provider"]).resolve()), region="us", expression_cache=None, dataset_cache=None)
with R.uri_context(str(source_card / "qlib_recorder")):
    recorder = R.get_recorder(experiment_id=src["experiment_id"], recorder_id=src["recorder_id"])
    daily = recorder.load_object("analysis/daily").copy()
    periods = recorder.load_object("analysis/periods")
    rolling = recorder.load_object("analysis/rolling126")
assert len(daily) == 944 and len(rolling) == 819
assert len(periods.loc[periods.kind.eq("quarter")]) == 16
assert periods.loc[periods.kind.eq("year") & periods.partial_boundary_period, "period"].astype(str).tolist() == ["2023", "2026"]
assert daily.trade_date.is_unique and daily.signal_date.is_unique
parts = ["cash_opportunity_log", "SOXL_selection_log", "Stage1_direct_fees_log", "timing_and_benchmark_residual_log"]
assert float((daily[parts].sum(axis=1) - daily.paired_log_excess).abs().max()) < 1e-12
calendar = pd.DatetimeIndex(D.calendar(start_time="2011-01-03", end_time="2026-10-09", freq="day"))
close = D.features(["SOXX"], ["$close"], start_time="2011-01-03", end_time="2026-10-09", freq="day")["$close"].droplevel("instrument").sort_index()
assert close.index.equals(calendar) and close.notna().all()
daily = daily.sort_values("trade_date").reset_index(drop=True)
measures = plan["definitions"]["factor_measures"] + plan["definitions"]["component_measures"]
rows = []
for i in range(819):
    segment = daily.iloc[i:i+126]
    rw = rolling.iloc[i]
    assert len(segment) == 126
    assert pd.Timestamp(segment.trade_date.iloc[0]) == pd.Timestamp(rw.first_trade)
    assert abs(float(segment.paired_log_excess.sum()) - float(rw.relative_log_excess)) < 1e-10
    signal = pd.Timestamp(segment.signal_date.iloc[0])
    loc = calendar.get_loc(signal)
    assert loc >= 20
    r5 = float(close.iloc[loc] / close.iloc[loc-5] - 1)
    r20 = float(close.iloc[loc] / close.iloc[loc-20] - 1)
    state = ("uptrend" if r5 >= 0 and r20 >= 0 else "rebound" if r5 >= 0 else "pullback" if r20 >= 0 else "decline")
    outcome = "weak" if rw.relative_log_excess < 0 else "good" if rw.relative_log_excess > 0 else "tie"
    row = {"first_trade": str(pd.Timestamp(rw.first_trade).date()), "year": int(pd.Timestamp(rw.first_trade).year), "start_state": state, "outcome": outcome}
    for name in plan["definitions"]["factor_measures"]:
        if name == "mean_raw_minus_cap":
            row[name] = float((segment.raw_nominal_beta - segment.own_state_beta_cap).mean())
        elif name == "cap_binding_fraction":
            row[name] = float(segment.cap_binding.mean())
        else:
            row[name] = float(segment[name.removeprefix("mean_")].mean())
    for name in plan["definitions"]["component_measures"]:
        row[name] = float(segment[name].sum())
    rows.append(row)
frame = pd.DataFrame(rows)
assert len(frame) == 819 and frame.outcome.isin(["weak", "good"]).all()
assert int(frame.outcome.eq("weak").sum()) == 564 and int(frame.outcome.eq("good").sum()) == 255
cells = []
for (state, year), group in frame.groupby(["start_state", "year"], sort=True):
    weak = group.loc[group.outcome.eq("weak")]
    good = group.loc[group.outcome.eq("good")]
    entry = {"start_state": state, "year": int(year), "weak_windows": len(weak), "good_windows": len(good), "matched": bool(len(weak) and len(good))}
    if entry["matched"]:
        entry["weak_minus_good"] = {m: float(weak[m].mean() - good[m].mean()) for m in measures}
        entry["weak_mean"] = {m: float(weak[m].mean()) for m in measures}
        entry["good_mean"] = {m: float(good[m].mean()) for m in measures}
    cells.append(entry)
matched = [c for c in cells if c["matched"]]
weights = np.array([min(c["weak_windows"], c["good_windows"]) for c in matched], dtype=float)
weighted = {m: float(np.average([c["weak_minus_good"][m] for c in matched], weights=weights)) for m in measures}
result = {
    "status": "read_only_factor_to_action_interval_diagnostic_no_policy",
    "plan_sha256": sha(plan_path),
    "source_recorder_id": src["recorder_id"],
    "source_NAS_sha256": src["nas_sha256"],
    "sessions": len(daily), "quarters": 16, "partial_endpoint_years": ["2023", "2026"],
    "rolling126_windows": len(frame), "weak_windows": int(frame.outcome.eq("weak").sum()), "good_windows": int(frame.outcome.eq("good").sum()),
    "matched_state_year_cells": len(matched), "total_state_year_cells": len(cells),
    "matched_cell_weight_sum": int(weights.sum()),
    "matched_weak_minus_good_weighted_descriptive": weighted,
    "cells": cells,
    "new_market_requests": 0, "new_model_fits": 0, "new_native_accounts": 0, "new_bootstraps": 0, "new_recorders": 0,
    "policy_inference": False, "independent_validation": False,
    "limits": ["Weak/good groups use future returns and are not tradable states", "126-session windows overlap heavily", "Own-state cap and funded beta are endogenous downstream account states"]
}
(card / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
print({k: result[k] for k in ["sessions", "rolling126_windows", "matched_state_year_cells", "matched_weak_minus_good_weighted_descriptive"]})


2026/10/24 15:52:56 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[13580:MainThread](2026-10-24 15:52:56,477) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[13580:MainThread](2026-10-24 15:52:56,478) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[13580:MainThread](2026-10-24 15:52:56,478) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'sessions': 944, 'rolling126_windows': 819, 'matched_state_year_cells': 12, 'matched_weak_minus_good_weighted_descriptive': {'mean_alpha_probability': 0.012353006196356502, 'mean_tail_probability': -0.016203701403636996, 'mean_raw_nominal_beta': -0.05245332100881761, 'mean_own_state_beta_cap': -0.07165289699778116, 'mean_ending_beta': -0.07487846236072836, 'mean_raw_minus_cap': 0.019199575988963668, 'cap_binding_fraction': 0.040895121882505735, 'cash_opportunity_log': -0.0430374727540491, 'SOXL_selection_log': -0.06779629826797061, 'Stage1_direct_fees_log': -0.00017664783536187783, 'paired_log_excess': -0.11123819712242455}}


## 中文结论

只读已验证的原 f696 A0 0bp Tiered Qlib 账户 Recorder 和同源 Qlib SOXX 日历，核对 944 日、16 季度、两端不完整年度及 819 个连续 126 日窗口。按窗口首交易日之前可得的 SOXX 5/20 日走势符号与首交易年配对；用未来窗口净超额只标记弱/好，绝不用于规则或阈值。0 新来源、拟合、账户、bootstrap。

12/16 个“事前状态 × 年份”格子同时有弱/好窗口；2023 年四格全部没有好窗口，不能从年内匹配得到2023的效应。在这12格中，弱窗口的平均 alpha 概率有11格**更高**，tail概率有9格更低，原始目标 beta 有9格更低，风险上限有8格更低，最终 beta 有8格更低。按每格较小组窗口数加权的描述差为 alpha +0.01235、tail -0.01620、原始 beta -0.05245、风险上限 -0.07165、最终 beta -0.07488。弱窗口现金机会分量平均差 -0.04304 对数点、SOXL 相对选择 -0.06780。原始 beta 减上限的均值差仅 +0.01920，12格中正负各6；不能把上限缺口单独当作信号。

这支持继续查“模型输出→配置意图→自身风险容量→实际持仓”的传递错位，而不是直接提高 alpha 权重、降低 tail 阈值或松绑 cap。alpha高却原始 beta 低，说明仅看一个头的分数无法解释配置；风险上限又进一步压低最终 beta。现金和 SOXL选择在弱区间均有损失，单现金层修补不够。所有差异都来自重叠、事后标记的复用历史，且2023年无年内好窗口；这不是因子有效性或因果证据，也不是新候选的0bp账户。

下一步先审计原39项及其头部输出怎样形成**配置意图**，并在事前同状态的好/差区间核对各输入的重复性与可用时间。只有找到一个能在训练期被否定、接收层和标签明确、与既有失败机制不同的新因子，才冻结信息门和完整 f696 自身账户。当前 `not_promoted`；SOXX v2目标未达成。
